# Libraries

In [1]:
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install --upgrade transformers
!pip -q install langdetect --break-system-packages #for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 74.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 88.4 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
MODEL_NAME       = "Llama-3.2-3B-Instruct-WANDA-25"
MODEL_PRETRAINED = "EdgeCompress01/Llama-3.2-3B-Instruct-WANDA"
SUB_FOLDER = "Models/25"
SEED             = 42

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Evaluations

**Configurations**

In [ ]:
# Task definition
TASKS = [
    Task("gsm8k",         "gsm8k_cot",    "math",       num_fewshot=8, batch_size=4, max_gen_toks=1024),
    Task("ifeval",        "ifeval",        "instruction_following", batch_size=4, max_gen_toks=1024),
    Task("arc_challenge", "arc_challenge", "reasoning",            batch_size=8),
    
    Task("wikitext", "wikitext", "perplexity", batch_size=1, apply_chat_template=False),
    Task("mmlu",    "mmlu",    "knowledge",  num_fewshot=5, batch_size=2, limit=1400),

    Task("gpqa_diamond", "gpqa_diamond_cot_zeroshot", "reasoning", batch_size=8, max_gen_toks=2048),
    Task("math-500", "minerva_math500", "math", num_fewshot=0, batch_size=8, max_gen_toks=2048),
]


# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"subfolder={SUB_FOLDER}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    #"enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [9]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "WARNING",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)

    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[MATH] gsm8k


2026-04-06:10:11:37 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:10:11:44 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-04-06:10:11:46 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 216341.58 examples/s]
2026-04-06:10:12:37 WARNING  [evaluator:333] Overwriting default num_fewshot of gsm8k_cot from 8 to 8
2026-04-06:10:12:37 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 1319/1319 [1:11:38<00:00,  3.26s/it]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-WANDA', 'subfolder': 'Models/25', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 8, batch_size: 4
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|---------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     8|exact_match|↑  |0.7278|±  |0.0123|
|         |       |strict-match    |     8|exact_match|↑  |0.6634|±  |0.0130|

Done

[INSTRUCTION_FOLLOWING] ifeval


2026-04-06:11:24:48 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:11:24:54 INFO     [_cli.run:376] Selected Tasks: ['ifeval']
2026-04-06:11:24:56 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating train split: 100%|██████████| 541/541 [00:00<00:00, 47094.73 examples/s]
2026-04-06:11:25:10 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 541/541 [1:18:17<00:00,  8.68s/it]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-WANDA', 'subfolder': 'Models/25', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 0, batch_size: 4
|Tasks |Version|Filter|n-shot|        Metric         |   |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|---|-----:|---|------|
|ifeval|      4|none  |     0|inst_level_loose_acc   |↑  |0.8177|±  |   N/A|
|      |       |none  |     0|inst_level_strict_acc  |↑  |0.7710|±  |   N/A|
|      |       |none  |     0|prompt_level_loose_acc |↑  |0.7320|±  |0.0191|
|      |       |none  |     0|prompt_level_strict_acc|↑  |0.6765|±  |0.0201|

Done

[REASONING] arc_challenge


...

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-WANDA', 'subfolder': 'Models/25', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 8
|    Tasks    |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-------------|------:|------|-----:|--------|---|-----:|---|-----:|
|arc_challenge|      1|none  |     0|acc     |↑  |0.4334|±  |0.0145|
|             |       |none  |     0|acc_norm|↑  |0.4420|±  |0.0145|

Done

[PERPLEXITY] wikitext


2026-04-06:12:46:32 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
2026-04-06:12:46:32 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-WANDA appears to be an instruct or chat variant but chat template is not applied. Recommend
        setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
Loading weights: 100%|██████████| 254/254 [00:01<00:00, 142.58it/s]
2026-04-06:12:46:46 WARNING  [api.task:729] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-06:12:46:46 WARNING  [api.task:741] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-04-06:12:46:46 WARNING  [api.task:729] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-06:12:46:46 WARNING  [api.task:741] [Task: wikitext] metric byte_perplexity is defined, but highe

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-WANDA', 'subfolder': 'Models/25', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 1
| Tasks  |Version|Filter|n-shot|    Metric     |   | Value |   |Stderr|
|--------|------:|------|-----:|---------------|---|------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  | 0.6976|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  | 1.6218|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |13.2707|±  |   N/A|

Done

[KNOWLEDGE] mmlu


2026-04-06:12:53:37 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-06:12:53:37 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:12:53:43 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 1818.39 examples/s]
2026-04-06:12:55:25 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
2026-04-06:12:55:25 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_anatomy from None to 5
2026-04-06:12:55:25 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_astronomy from None to 5
2026-04-06:12:55:25 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_biology from None to 5
2026-04-06:12:55:25 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_chemistry from None to 5
2026-04-06:12:55:25 WARNING  [evaluato

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-WANDA', 'subfolder': 'Models/25', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: 1400.0, num_fewshot: 5, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.6057|±  |0.0040|
| - humanities                          |      2|none  |      |acc   |↑  |0.5666|±  |0.0071|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.4286|±  |0.0443|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.7394|±  |0.0343|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |0.7353|±  |0.0310|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |0.7764|±  |0.0271|
|  - international_law                 

# Save reports

In [10]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)